# Codex Python SDK Walkthrough

Public SDK surface only (`codex_app_server` root exports).

In [ ]:
# Cell 1: bootstrap current kernel with local SDK + dependencies
import os
import subprocess
import sys
from pathlib import Path

try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(Path.home()))

candidates = [
    Path.cwd(),
    Path.home() / '.openclaw/workspace/codex-py-sdk/sdk/python',
]
repo_python_dir = next((p for p in candidates if (p / 'pyproject.toml').exists()), None)
if repo_python_dir is None:
    raise RuntimeError('Could not locate sdk/python. Set repo_python_dir manually.')

src_dir = repo_python_dir / 'src'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(repo_python_dir)])
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print('Kernel:', sys.executable)
print('SDK source:', src_dir)

In [ ]:
# Cell 2: imports (public only)
from codex_app_server import (
    AsyncCodex,
    Codex,
    ImageInput,
    LocalImageInput,
    TextInput,
    ThreadListParams,
    ThreadStartParams,
    retry_on_overload,
)

In [ ]:
# Cell 3: simple sync conversation
with Codex() as codex:
    thread = codex.thread_start(ThreadStartParams(model='gpt-5'))
    turn = thread.turn(TextInput('Explain gradient descent in 3 bullets.'))
    result = turn.run()

    print('server:', codex.metadata)
    print('status:', result.status)
    print(result.text)

In [ ]:
# Cell 4: multi-turn continuity in same thread
with Codex() as codex:
    thread = codex.thread_start(ThreadStartParams(model='gpt-5'))

    first = thread.turn(TextInput('Give a short summary of transformers.')).run()
    second = thread.turn(TextInput('Now explain that to a high-school student.')).run()

    print('first status:', first.status)
    print('second status:', second.status)
    print('second text:', second.text)

In [ ]:
# Cell 5: thread lifecycle APIs
with Codex() as codex:
    thread = codex.thread_start(ThreadStartParams(model='gpt-5'))

    listing = codex.thread_list(ThreadListParams(limit=10))
    reading = thread.read(include_turns=False)

    print('thread list fetched:', len(listing.data))
    print('thread read fetched for:', reading.thread.id)

In [ ]:
# Cell 6: multimodal with remote image (best effort)
with Codex() as codex:
    thread = codex.thread_start(ThreadStartParams(model='gpt-5'))
    try:
        result = thread.turn([
            TextInput('What do you see in this image? 3 bullets.'),
            ImageInput('https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg'),
        ]).run()
        print('status:', result.status)
        print(result.text)
    except Exception as e:
        print('Remote image path failed in this environment:', type(e).__name__, e)

In [ ]:
# Cell 7: multimodal with local image
from base64 import b64decode
from pathlib import Path

sample_img = Path.cwd() / 'sample.png'
if not sample_img.exists():
    sample_img.write_bytes(
        b64decode('iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mP8/x8AAwMCAO7Z4xQAAAAASUVORK5CYII=')
    )

with Codex() as codex:
    thread = codex.thread_start(ThreadStartParams(model='gpt-5'))
    result = thread.turn([
        TextInput('Describe this local image in 2 bullets.'),
        LocalImageInput(str(sample_img)),
    ]).run()

    print('status:', result.status)
    print(result.text)

In [ ]:
# Cell 8: retry-on-overload pattern
with Codex() as codex:
    thread = codex.thread_start(ThreadStartParams(model='gpt-5'))

    result = retry_on_overload(
        lambda: thread.turn(TextInput('List 5 failure modes in distributed systems.')).run(),
        max_attempts=3,
        initial_delay_s=0.25,
        max_delay_s=2.0,
    )

    print('status:', result.status)
    print(result.text)

In [ ]:
# Cell 9: async parity usage (same flow as sync)
import asyncio

async def async_demo():
    async with AsyncCodex() as codex:
        thread = await codex.thread_start(ThreadStartParams(model='gpt-5'))
        turn = await thread.turn(TextInput('One sentence about scientific reproducibility.'))
        result = await turn.run()

        print('server:', codex.metadata)
        print('status:', result.status)
        print(result.text)

await async_demo()

In [ ]:
# Cell 10: async event streaming
import asyncio

async def async_stream_demo():
    async with AsyncCodex() as codex:
        thread = await codex.thread_start(ThreadStartParams(model='gpt-5'))
        turn = await thread.turn(TextInput('Write a short haiku about compilers.'))

        async for event in turn.stream():
            print(event.method, event.payload)

await async_stream_demo()